# E005 — Feature-group ablations

**Hypotheses:** H2, H3, H4, H7 ([docs/hypotheses.md](../../docs/hypotheses.md))

**Question:** Which feature groups earn their place in the model?

**Method:** 3-seed DCN-v2 ensembles on the validation day, each identical to the all-features reference except for one change; paired hour-block bootstrap of the log-loss difference (raw logits, because the calibrator is fitted on validation).

In [1]:
from charade.analysis.experiment import setup

ctx = setup()
f"mode: {'smoke (fixture, tiny model)' if ctx.smoke else 'full data'}"

'mode: full data'

In [2]:
import matplotlib.pyplot as plt
import polars as pl

from charade.models import ablate

table = ablate.run(out=ctx.reports / "models", data_dir=ctx.data_dir)
table.with_columns(pl.col(pl.Float64).round(5))

variant,val_ne,delta_logloss,ci_low,ci_high
str,f64,f64,f64,f64
"""all features""",0.87199,0.0,0.0,0.0
"""- character_meta""",0.8792,0.00324,0.00271,0.00373
"""- character_id""",0.87195,-0.00002,-0.00023,0.00019
"""- all character features""",0.89511,0.01037,0.00954,0.01137
"""- user_history""",0.87394,0.00088,0.00038,0.00134
…,…,…,…,…
"""- device""",0.87633,0.00195,0.00147,0.00259
"""+ text (qwen3)""",0.87379,0.00081,0.00056,0.00107
"""+ text (tfidf)""",0.87369,0.00076,0.00056,0.00097


In [3]:
rows = table.filter(pl.col("variant") != "all features")
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(
    rows["delta_logloss"],
    range(rows.height),
    xerr=[rows["delta_logloss"] - rows["ci_low"], rows["ci_high"] - rows["delta_logloss"]],
    fmt="o",
)
ax.axvline(0, color="grey", lw=1)
ax.set_yticks(range(rows.height), rows["variant"].to_list())
ax.set_xlabel("validation log loss vs all features (positive = worse)")
plt.tight_layout()
plt.show()

/tmp/ipykernel_865567/2907948984.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Reading
- H2: removing metadata costs far more than removing the character ID, which costs nothing.
- H3: text features make the model worse, consistent with E002.
- H4 partly refuted: user history helps, but less than device and character metadata.
- H7: conversation features are within noise.
- Deltas of +/-0.0003 (e.g. ID dropout, the shipped variant) are within seed noise, which this
  bootstrap over hours does not include.

## Conclusion
Character metadata is worth +0.0032 log loss and user history +0.0009; character ID and conversation features are within noise and were dropped; text features hurt (+0.0008).

**Decision:** Ship without character ID and conversation groups; reject text; keep metadata, device and user history.